In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.globals import set_debug, set_verbose


from pydantic import BaseModel


from dotenv import load_dotenv
import os

load_dotenv()

print("Tracing:", os.getenv("LANGSMITH_TRACING"))
print("Project:", os.getenv("LANGSMITH_PROJECT"))
print("Endpoint:", os.getenv("LANGSMITH_ENDPOINT"))

Tracing: true
Project: excerices
Endpoint: https://api.smith.langchain.com


### Task 1 

In [2]:
class RecommandSchema(BaseModel):
    genre: str
    movies: list[str]


model = GoogleGenerativeAI(model="gemini-3.5-flash-lite")
parser = PydanticOutputParser(pydantic_object=RecommandSchema)

In [3]:
prompt = ChatPromptTemplate.from_template(
    """
You are Movie recommandator in netflix
your task is recommande movie based on user search 
your out format must match with schema
{format_instructions}

Do not add unrelated movies 
Do not give priority to new movies over user's requirements
If user's requirement not match with you simply genrate null in all value.
<!-- and make sure if you not found any movie than it's genre also None and movies also None -->

Examples of output format: 
{{ "genre": "Horror", "movies": [ "The Conjuring", "Hereditary", "Insidious", "Sinister", "The Exorcist" ] }} 
{{ "genre": "Family", "movies": [ "Toy Story", "Finding Nemo", "The Lion King", "Up", "Moana" ] }} 
If no movie matches: {{ "genre": null, "movies": [] }}

search query:- 
{user_query}
"""
).partial(format_instructions=parser.get_format_instructions())

In [18]:
print(prompt)

input_variables=['user_query'] input_types={} partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"genre": {"title": "Genre", "type": "string"}, "movies": {"items": {"type": "string"}, "title": "Movies", "type": "array"}}, "required": ["genre", "movies"]}\n```'} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['format_instructions', 'user_query'], input_types={}, partial_variables={}, template='\nYou are Movie recommandator in netflix\nyour task is recommande movie based on user search \nyour out format

In [4]:
chain = prompt | model | parser

In [5]:
result = await chain.ainvoke(
    {"user_query": "watch movie where 2 heroes and focus on fight with others."}
)

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [6]:
result

RecommandSchema(genre='Action', movies=['Bad Boys', 'Rush Hour', 'Hobbs & Shaw', 'The Nice Guys', 'Step Brothers'])

### Task 2

In [52]:
class CustomPgVectorRetriever(BaseRetriever):
    tenant_id: str
    connection_string: str
    embedding_model: object
    top_k: int = 5

    # Required by BaseRetriever
    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager=None,
    ) -> list[Document]:
        raise NotImplementedError("This retriever is async. Use 'await retriever.ainvoke(query)'")

    async def _aget_relevant_documents(
        self,
        query: str,
        *,
        run_manager=None,
    ) -> list[Document]:
        # 1. Convert query into embedding

        # 2. Search existing pgvector table

        # 3. Convert database rows to LangChain Documents
        return [
            Document(
                page_content="Excercise",
                metadata={
                    "chunk_id": "Excercise",
                    "similarity_score": 0.0,
                },
            )
        ]

In [63]:
retriever = CustomPgVectorRetriever(
    tenant_id="YOUR_TENANT_ID",
    connection_string="DATABASE_CONNECTION_CONVERSATION_URL",
    embedding_model="YOUR_EMBEDDING_MODEL",
    top_k=5,
)

documents = await retriever.ainvoke("Question")


rag_chain = retriever | prompt | model  # actual query, prompt, model logic change

### Task 3

include in task 1

### Task 4

In [7]:
prompt_history = ChatPromptTemplate.from_messages(
    [
        ("system", "you are chat assistant give answers based on user query"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{user_query}"),
    ]
)

chain_for_history = prompt_history | model

In [8]:
store: dict[str, InMemoryChatMessageHistory] = {}


def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [9]:
chat_with_history = RunnableWithMessageHistory(
    chain_for_history, get_history, input_messages_key="user_query", history_messages_key="history"
)

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
response = chat_with_history.invoke(
    {"user_query": "Give me my result"}, config={"configurable": {"session_id": "user_1"}}
)

print(response)

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Here is your overall performance summary:

* **Round 1 (General GK):** 2 out of 3 correct
* **Round 2 (Indian GK):** 3 out of 3 correct

**Total Score:** 5 out of 6 correct! 

Overall, you did a fantastic job, especially dominating the India-specific round. Let me know if you want to play again!


In [100]:
get_history(session_id="user_1")

InMemoryChatMessageHistory(messages=[HumanMessage(content='ask me 3 questions of GK', additional_kwargs={}, response_metadata={}), AIMessage(content="Here are 3 General Knowledge questions for you:\n\n1. What is the longest river in the world?\n2. Which planet is known as the Red Planet?\n3. Who painted the Mona Lisa? \n\nTake your time, and let me know your answers whenever you're ready!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='1. amazon, 2. juipter, 3. idk. here is my answer correct it if wrong', additional_kwargs={}, response_metadata={}), AIMessage(content="Here are the corrections for your answers:\n\n1. **Your answer:** Amazon\n   * **Correction:** Incorrect. The longest river in the world is the **Nile** River (though the Amazon is the largest by water volume).\n\n2. **Your answer:** Juipter\n   * **Correction:** Incorrect. The Red Planet is **Mars**. (Jupiter is known as the Gas Giant).\n\n3. **Your answer:** idk

### Task 5

In [16]:
bad_model = RunnableLambda(lambda _: "This is not valid JSON and does not match the schema")

failing_chain = prompt | bad_model | parser

In [17]:
try:
    result = failing_chain.invoke({"user_query": "Recommend horror movies"})

    print(result)

except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR:", e)

ERROR TYPE: OutputParserException
ERROR: Invalid json output: This is not valid JSON and does not match the schema
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


### Task 6

In [131]:
# LCEL chain
lcel_chain = prompt | model | parser

user_query = "Recommend some supernatural horror movies"

lcel_response = lcel_chain.invoke({"user_query": user_query})

print("LCEL RESPONSE:")
print(lcel_response)

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


LCEL RESPONSE:
genre='Supernatural Horror' movies=['The Conjuring', 'Insidious', 'Sinister', 'Hereditary', 'The Exorcist']


In [118]:
# Function


def movie_recommendation(user_query: str):
    # Step 1: Format prompt
    formatted_prompt = prompt.invoke({"user_query": user_query})

    # Step 2: Invoke model
    model_response = model.invoke(formatted_prompt)

    # Step 3: Parse model response
    parsed_response = parser.invoke(model_response)

    return parsed_response


python_response = movie_recommendation("Recommend some supernatural horror movies")

print("\nPYTHON RESPONSE:")
print(python_response)

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



PYTHON RESPONSE:
genre='Horror' movies=['The Conjuring', 'Insidious', 'Sinister', 'Hereditary', 'The Exorcist']


In [119]:
# for debugging


def movie_recommendation_debug(user_query: str):
    print("\n========== STEP 1: PROMPT ==========")

    formatted_prompt = prompt.invoke({"user_query": user_query})

    print(formatted_prompt)

    print("\n========== STEP 2: MODEL ==========")

    model_response = model.invoke(formatted_prompt)

    print(model_response)

    print("\n========== STEP 3: PARSER ==========")

    parsed_response = parser.invoke(model_response)

    print(parsed_response)

    return parsed_response


debug_response = movie_recommendation_debug("Recommend some supernatural horror movies")


========== STEP 1: PROMPT ==========
messages=[HumanMessage(content='\nYou are Movie recommandator in netflix\nyour task is recommande movie based on user search \nyour out format must match with schema\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"genre": {"title": "Genre", "type": "string"}, "movies": {"items": {"type": "string"}, "title": "Movies", "type": "array"}}, "required": ["genre", "movies"]}\n```\n\nDo not add unrelated movies \nDo not give priority to new movies over user\'s requirements\nIf user\'s requirement not match with you simply genrate null i

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


```json
{
  "genre": "Horror",
  "movies": [
    "The Conjuring",
    "Insidious",
    "Sinister",
    "The Exorcist",
    "Hereditary"
  ]
}
```

========== STEP 3: PARSER ==========
genre='Horror' movies=['The Conjuring', 'Insidious', 'Sinister', 'The Exorcist', 'Hereditary']


### Task 7

In [129]:
set_verbose(True)

lcel_chain = prompt | model | parser

user_query = "Recommend some supernatural horror movies"

lcel_response = lcel_chain.invoke({"user_query": user_query})

print("LCEL RESPONSE:")
print(lcel_response)

set_verbose(False)

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


LCEL RESPONSE:
genre='Horror' movies=['The Conjuring', 'Insidious', 'Sinister', 'The Exorcist', 'Hereditary']


In [15]:
set_debug(True)
lcel_chain = prompt | model | parser

user_query = "Recommend some supernatural horror movies"

lcel_response = lcel_chain.invoke({"user_query": user_query})

print("LCEL RESPONSE:")
print(lcel_response)

set_debug(False)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "user_query": "Recommend some supernatural horror movies"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "user_query": "Recommend some supernatural horror movies"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:GoogleGenerativeAI] Entering LLM run with input:
{
  "prompts": [
    "Human: \nYou are Movie recommandator in netflix\nyour task is recommande movie based on user search \nyour out format must match with schema\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is

/home/harshilk/genai-engineer-journey/module-10-langchain/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[llm/end] [chain:RunnableSequence > llm:GoogleGenerativeAI] [1.03s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "```json\n{\n  \"genre\": \"Supernatural Horror\",\n  \"movies\": [\n    \"The Conjuring\",\n    \"Insidious\",\n    \"Sinister\",\n    \"Hereditary\",\n    \"The Exorcist\"\n  ]\n}\n```",
        "generation_info": {
          "finish_reason": "STOP",
          "model_name": "gemini-3.5-flash-lite",
          "safety_ratings": [],
          "usage_metadata": {
            "input_tokens": 380,
            "output_tokens": 61,
            "total_tokens": 441,
            "input_token_details": {
              "cache_read": 0
            }
          }
        },
        "type": "Generation"
      }
    ]
  ],
  "llm_output": null,
  "run": null,
  "type": "LLMResult"
}
[chain/start] [chain:RunnableSequence > parser:PydanticOutputParser] Entering Parser run with input:
{
  "input": "```json\n{\n  \"genre\": \"Supernatural Horror\",\n  \"movies